# 04 · Embeddings, prototypes, k-NN and autoencoders (guided)

**Style A: syntax gaps.** 8 gaps (`# TODO(gN)` + `____`), each followed by a check.
Solutions: `../solutions/guided/04_embeddings_prototypes.md`.

**Traces to** `notes/ioai-task-patterns.md` P4, P6. In T24-O-BOBAI you received **cached embeddings** plus a
frozen 5-way `Linear` head and had to reach 7 classes using only "averages and distances". The reference used
k-NN on the embeddings. T24-O-HYPER allowed unsupervised extractors (autoencoders). The syllabus lists "Data
Embeddings (Both)" and "Autoencoders (Practice)".

Runtime: about 1 minute on a T4 (the ResNet18 forward passes dominate).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from datasets import load_dataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

cifar = load_dataset("uoft-cs/cifar10")
def balanced(split, per_class, seed):
    labels = np.array(split["label"]); rng = np.random.default_rng(seed)
    idx = np.concatenate([rng.choice(np.flatnonzero(labels == c), per_class, replace=False) for c in range(10)])
    return split.select(idx)
train, test = balanced(cifar["train"], 200, 0), balanced(cifar["test"], 100, 1)
print(train, test.features["label"].names)

## 1. A frozen pretrained backbone and its matching preprocessing

torchvision weight enums carry the exact preprocessing the model was trained with.

In [ ]:
# TODO(g1): the default ImageNet weights enum for ResNet18
weights = ____
backbone = models.resnet18(weights=weights).to(device).eval()
# TODO(g2): the preprocessing pipeline that belongs to these weights
preprocess = ____
print(preprocess)

In [ ]:
x = preprocess(torch.randint(0, 256, (2, 3, 32, 32), dtype=torch.uint8))
assert x.shape == (2, 3, 224, 224) and x.dtype == torch.float32 and x.min() < 0, "resize + normalise expected"
print("g1, g2 ok")

## 2. Getting the 512-d embedding: replace the head, or use a forward hook

In [ ]:
def to_batch(pil_images):
    return torch.stack([torch.from_numpy(np.array(im.convert("RGB"))).permute(2, 0, 1) for im in pil_images])

@torch.no_grad()
def embed(dataset, model, bs=250):
    out = []
    for i in range(0, len(dataset), bs):
        xb = preprocess(to_batch(dataset[i:i + bs]["img"])).to(device)
        out.append(model(xb).cpu())
    return torch.cat(out)

# TODO(g3): swap the classification layer for a no-op so the model returns the pooled 512-d features
backbone.fc = ____
E_tr, E_te = embed(train, backbone), embed(test, backbone)
y_tr, y_te = torch.tensor(train["label"]), torch.tensor(test["label"])
print(E_tr.shape, E_te.shape)

In [ ]:
assert E_tr.shape == (2000, 512)
print("g3 ok")

A forward hook reads an intermediate activation without changing the model. The hook signature is
`(module, inputs, output)`.

In [ ]:
store = []
# TODO(g4): on layer4, store the spatial mean of its output, shape (B, 512)
handle = backbone.layer4.register_forward_hook(____)
with torch.no_grad():
    _ = backbone(preprocess(to_batch(train[:8]["img"])).to(device))
handle.remove()

In [ ]:
# GPU convs differ by ~1e-3 between batch sizes (TF32 / cuDNN algorithm choice), hence the loose tolerance
assert torch.allclose(store[0], E_tr[:8], rtol=1e-2, atol=1e-2), "hook output should equal the avgpool features"
print("g4 ok")

## 3. Class prototypes (nearest class mean) with cosine similarity

In [ ]:
# TODO(g5): L2-normalise every embedding (row-wise)
Z_tr, Z_te = ____
# TODO(g6): one prototype per class: the mean normalised embedding of that class -> (10, 512)
P = ____
# TODO(g7): predict the class whose (normalised) prototype has the highest cosine similarity
pred = ____
print("nearest-prototype accuracy:", accuracy_score(y_te, pred))

In [ ]:
assert torch.allclose(Z_tr.norm(dim=1), torch.ones(len(Z_tr)), atol=1e-5)
assert P.shape == (10, 512) and torch.allclose(P[3], Z_tr[y_tr == 3].mean(0))
assert accuracy_score(y_te, pred) > 0.7
print("g5, g6, g7 ok")

In [ ]:
knn = KNeighborsClassifier(n_neighbors=10, metric="cosine").fit(E_tr, y_tr)
print("10-NN (cosine) accuracy:", accuracy_score(y_te, knn.predict(E_te)))

## 4. An autoencoder: compress 512 → 32 without labels

In [ ]:
encoder = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 32)).to(device)
decoder = nn.Sequential(nn.Linear(32, 128), nn.ReLU(), nn.Linear(128, 512)).to(device)
opt = torch.optim.Adam([*encoder.parameters(), *decoder.parameters()], lr=1e-3)
Xall = torch.cat([Z_tr, Z_te]).to(device)             # unsupervised: test features may be used, labels may not
for step in range(1500):
    xb = Xall[torch.randint(0, len(Xall), (256,), device=device)]
    # TODO(g8): reconstruction loss: decode(encode(x)) should match x
    loss = ____
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0: print(step, round(loss.item(), 5))
with torch.no_grad():
    C_tr, C_te = encoder(Z_tr.to(device)).cpu(), encoder(Z_te.to(device)).cpu()
print("10-NN on 32-d codes:", accuracy_score(y_te, KNeighborsClassifier(10).fit(C_tr, y_tr).predict(C_te)))

In [ ]:
with torch.no_grad():
    rec = F.mse_loss(decoder(encoder(Xall)), Xall).item()
assert rec < 0.5 * Xall.var(0).sum().item() / 512, "the autoencoder should explain most of the variance"
print("g8 ok, all gaps done")

## 5. The T24-O-BOBAI trick: extend a frozen 5-way head without new parameters

A `Linear(512, 5)` head knows classes 0–4 (airplane … deer). Classes 5 and 6 arrive with **20 examples each**.
We may not train anything new, but we may use **means and distances**:
1. Route: if the nearest of the 7 prototypes is a new class, predict it.
2. Otherwise trust the frozen head.

In [ ]:
old = y_tr < 5
head = nn.Linear(512, 5)
opt = torch.optim.Adam(head.parameters(), lr=1e-2)
for _ in range(300):                                      # the "deployed" head, trained once on old data
    loss = F.cross_entropy(head(Z_tr[old]), y_tr[old]); opt.zero_grad(); loss.backward(); opt.step()
head.requires_grad_(False)

few = torch.cat([torch.nonzero(y_tr == c).flatten()[:20] for c in (5, 6)])
P7 = F.normalize(torch.cat([P[:5], torch.stack([Z_tr[few][y_tr[few] == c].mean(0) for c in (5, 6)])]), dim=1)
mask7 = y_te < 7
route = (Z_te[mask7] @ P7.T).argmax(1)
pred7 = torch.where(route >= 5, route, head(Z_te[mask7]).argmax(1))
print("7-way macro F1:", round(f1_score(y_te[mask7], pred7, average="macro"), 3))